In [1]:
import napari
import scipy as sp
import numpy as np
import tifffile
import scipy.ndimage as ndi
from skimage.feature import peak_local_max
import os
import glob
import cv2
import skimage as ski
import dask
import plotly.express as px
import pandas as pd
import plotly.graph_objs as go
import nd2

# Setup Notebook

In [2]:
viewer = napari.Viewer()

In [3]:
scale = [0.50, 0.065, 0.065]

# Utility Functions

In [4]:
def backsub(inp, radius=20):
    filterSize =(radius, radius)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                    filterSize)
    blurred = cv2.GaussianBlur(inp, (5, 5), 0)
    tophat_img = cv2.morphologyEx(blurred,
                                cv2.MORPH_TOPHAT,
                                kernel)
    rtn = inp.astype(np.single) - (blurred-tophat_img)
    rtn = np.clip(rtn, 0, np.inf)
    return rtn
def backsub_3d(inp, radius=20):
    shape = inp.shape
    reshaped = inp.reshape(-1, shape[-2], shape[-1])
    # process = [dask.delayed(backsub)(i, radius) for i in reshaped]
    # rslt = np.array(dask.compute(*process))
    process = [backsub(i, radius) for i in reshaped]
    rslt = np.array(process)
    rslt = rslt.reshape(shape)
    return rslt

In [5]:
def filter_label_size(labels, min_size=10, max_size=100):
    label_sizes = np.bincount(labels.ravel())
    too_small = label_sizes < min_size
    too_large = label_sizes > max_size
    labels[np.isin(labels, np.where(too_small | too_large))] = 0
    labels = ski.segmentation.relabel_sequential(labels)[0]
    return labels

In [6]:
def find_peaks(img, threshold=0.1, display=False):
    #LoG = -ndi.gaussian_laplace(img/1000, sigma=[2,4,4])
    #LoG = -ndi.gaussian_laplace(img/1000, sigma=[1,2,2])
    
    LoG = -ndi.gaussian_laplace(img/1000, sigma=[3,5,5])
    max_peaks = peak_local_max(LoG, min_distance=1, threshold_rel=threshold)
    if display:
        viewer.add_image(LoG, blending='additive', colormap='magenta', scale=scale)
        viewer.add_points(max_peaks, n_dimensional=True, size=6, scale=scale, name='AllPeaks')
    return max_peaks

def pair_finder(peaks, max_dist=6, scale=[0.12, 0.04, 0.04]):
    scaled_peaks = peaks.copy().astype(float)
    scaled_peaks[:,0] = scaled_peaks[:,0] * scale[0]
    scaled_peaks[:,1] = scaled_peaks[:,1] * scale[1]
    scaled_peaks[:,2] = scaled_peaks[:,2] * scale[2]
    
    d_matrix = sp.spatial.distance.squareform(sp.spatial.distance.pdist(scaled_peaks))
    d_matrix[d_matrix==0] = 10000
    
    min_pos = np.argmin(d_matrix, axis=0)
    distances = np.min(d_matrix, axis=0)
    sorte = np.argsort(distances)
    
    return_distances = []
    lst = []
    for a in np.arange(0,len(sorte)):
        if distances[sorte[a]]<max_dist:
            if any([all (peaks[sorte[a]]==b) for b in lst]) | any([all(peaks[min_pos][sorte[a]]==b) for b in lst]):
                lst
            elif np.abs(peaks[sorte[a]][0]-peaks[min_pos][sorte[a]][0])>6:
                lst
            else:
                lst.append(peaks[sorte[a]])
                lst.append(peaks[min_pos][sorte[a]])
                return_distances.append(distances[sorte[a]])
    return lst, return_distances

def find_peaks_in_file(img, display=False, threshold=0.16, viewer=None, peak_channel=1, max_dist=1.2):
    peaks = find_peaks(img[:,peak_channel,:,:], threshold=threshold, display=display)
    filtered_peaks, distances = pair_finder(peaks, max_dist=max_dist, scale=scale)

    if display:
        viewer.add_points(filtered_peaks[::2], n_dimensional=True, size=10, scale=scale, name='FilteredPeaks', face_color='magenta')
        viewer.add_points(filtered_peaks[1::2], n_dimensional=True, size=10, scale=scale, name='FilteredPeaks', face_color='yellow')
    return filtered_peaks, distances

# Process Data

In [8]:
fnames = glob.glob('*/*/*.nd2')

In [11]:
def get_peaks_merge_nuclei(img, nuclei_df, nuclei_labels, peak_channel=0, display=False, viewer=None):
    peaks, _ = find_peaks_in_file(img, display=display, viewer=viewer, threshold=0.16, peak_channel=peak_channel, max_dist=10)
    indices = np.arange(1, len(peaks)+1)
    peak_labels = img[:,0] * 0
    pks = np.array(peaks)
    peak_labels[pks[:,0], pks[:,1], pks[:,2]] = indices

    smoothed = ndi.gaussian_filter(img[:,peak_channel].astype(np.float32), sigma=[1,5,5])
    smoothed = smoothed - np.percentile(smoothed, 50)
    smoothed = np.clip(smoothed, 0, np.inf)
    puncta_df = pd.DataFrame(ski.measure.regionprops_table(peak_labels, intensity_image=smoothed, properties=['label', 'mean_intensity', 'centroid'])).rename(columns={'centroid-0': 'z', 'centroid-1': 'y', 'centroid-2': 'x'})
    puncta_nucleus_df = pd.DataFrame(ski.measure.regionprops_table(peak_labels, intensity_image=nuclei_labels, properties=['label', 'mean_intensity'])).rename(columns={'mean_intensity': 'nucleus_label'})
    puncta_nucleus_df['nucleus_label'] = puncta_nucleus_df['nucleus_label'].round().astype(int)
    puncta_df = puncta_df.merge(puncta_nucleus_df, on='label')
    puncta_df = puncta_df.merge(nuclei_df, left_on='nucleus_label', right_on='label', suffixes=('_puncta', '_nucleus'))
    puncta_df.loc[:,['z_puncta', 'y_puncta', 'x_puncta']] = puncta_df.loc[:,['z_puncta', 'y_puncta', 'x_puncta']] * scale
    puncta_df.loc[:,['z_nucleus', 'y_nucleus', 'x_nucleus']] = puncta_df.loc[:,['z_nucleus', 'y_nucleus', 'x_nucleus']] * scale
    puncta_df['area'] = puncta_df['area'] * scale[0] * scale[1] * scale[2]

    puncta_df['puncta_count'] = puncta_df.groupby('label_nucleus')['label_puncta'].transform(len)
    puncta_df = puncta_df[puncta_df['puncta_count'] == 2]
    puncta_df['dR'] = np.linalg.norm(puncta_df[['z_puncta', 'y_puncta', 'x_puncta']].values - puncta_df[['z_nucleus', 'y_nucleus', 'x_nucleus']].values, axis=1)

    if display:
        viewer.add_image(smoothed, scale=scale, name=f'Puncta Channel {peak_channel} Smoothed')

    puncta_df = puncta_df.sort_values(by=['label_nucleus', 'mean_intensity'], ascending=[True, False])
    return puncta_df

def process_fname(fname, display=False, viewer=None):
    img = nd2.imread(fname)

    # Segment the nuclei
    ds_dapi = ski.transform.downscale_local_mean(img[:,2], (1,8,8))
    back_subbed = backsub_3d(ds_dapi, radius=50)
    # back_subbed = ds_dapi - np.percentile(ds_dapi, 50)
    # back_subbed = np.clip(back_subbed, 0, np.inf)

    smoothed = ndi.gaussian_filter(back_subbed, sigma=[1,2,2])
    otsu = 1.0 * ski.filters.threshold_otsu(smoothed)
    binary = smoothed > otsu
    filled = ndi.binary_fill_holes(binary)
    labels = ski.measure.label(filled)
    labels = ski.transform.resize(labels, img[:,2].shape, order=0, preserve_range=True).astype(int)
    nuclei_df = pd.DataFrame(ski.measure.regionprops_table(labels, properties=['label', 'centroid', 'area'])).rename(columns={'centroid-0': 'z', 'centroid-1': 'y', 'centroid-2': 'x'})

    if display:
        viewer.add_image(img, channel_axis=1, scale=scale)
        viewer.add_labels(labels, scale=scale, name='Nuclei Labels')

    # Get the two peak dfs
    puncta_df_c0 = get_peaks_merge_nuclei(img, nuclei_df, labels, peak_channel=0, display=display, viewer=viewer)
    puncta_df_c0['channel'] = 'C0'
    puncta_df_c0['fname'] = fname
    puncta_df_c1 = get_peaks_merge_nuclei(img, nuclei_df, labels, peak_channel=1, display=display, viewer=viewer)
    puncta_df_c1['channel'] = 'C1'
    puncta_df_c1['fname'] = fname

    return pd.concat([puncta_df_c0, puncta_df_c1], ignore_index=True)


In [14]:
cdf = process_fname(fnames[30], display=True, viewer=viewer)

In [ ]:
reprocess = False
if reprocess:
    process = [dask.delayed(process_fname)(fname) for fname in fnames]
    df = pd.concat(dask.compute(*process), ignore_index=True)
    df.to_csv('Results.csv')


# Analyze Results

In [36]:
df = pd.read_csv('Results.csv')
df['Brightest'] = df.groupby(['fname', 'channel', 'nucleus_label'])['mean_intensity'].transform(lambda x: x==x.max())
df['Chromosome'] = df['fname'].str.split('-').str[0]
df['rep'] = df['fname'].str.split('\\').str[1]
df = df.rename(columns={'area': 'volume'})
df = df[(df['volume'] < 1500) & (df['volume'] > 300)]
df

,Unnamed: 0,label_puncta,mean_intensity,z_puncta,y_puncta,x_puncta,nucleus_label,label_nucleus,z_nucleus,y_nucleus,x_nucleus,volume,puncta_count,dR,channel,fname,Brightest,Chromosome,rep
0,0,8,1526.487549,9.0,92.495,24.635,1,1,7.156098,91.458123,22.113932,417.9032,2,3.291029,C0,Cen18-488_Cen6-561\rep1\FISH001.nd2,True,Cen18,rep1
1,1,7,1242.533569,6.5,89.310,22.815,1,1,7.156098,91.458123,22.113932,417.9032,2,2.352954,C0,Cen18-488_Cen6-561\rep1\FISH001.nd2,False,Cen18,rep1
2,2,10,2778.539795,8.5,61.815,46.475,2,2,9.497959,57.205714,49.720525,885.9318,2,5.724935,C0,Cen18-488_Cen6-561\rep1\FISH001.nd2,True,Cen18,rep1
3,3,9,2330.579590,10.5,58.110,44.135,2,2,9.497959,57.205714,49.720525,885.9318,2,5.746295,C0,Cen18-488_Cen6-561\rep1\FISH001.nd2,False,Cen18,rep1
4,4,4,1967.144531,11.0,82.420,98.540,3,3,9.684999,80.057901,94.289121,974.8596,2,5.037729,C0,Cen18-488_Cen6-561\rep1\FISH001.nd2,True,Cen18,rep1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
829,829,2,5986.328613,5.5,102.245,113.425,8,8,6.218825,107.111990,111.544271,1160.7765,2,5.267015,C0,Cen7-488_Cen11-561\rep2\FISH010.nd2,False,Cen7,rep2
830,830,4,3898.404053,5.0,101.400,20.410,10,10,6.354626,102.045199,19.365972,318.5312,2,1.827920,C0,Cen7-488_Cen11-561\rep2\FISH010.nd2,True,Cen7,rep2
831,831,3,3079.965332,5.5,103.155,12.220,10,10,6.354626,102.045199,19.365972,318.5312,2,7.281961,C0,Cen7-488_Cen11-561\rep2\FISH010.nd2,False,Cen7,rep2
832,832,2,21464.886719,6.5,52.845,21.710,3,3,5.775007,55.958232,18.880464,923.8216,2,4.268969,C1,Cen7-488_Cen11-561\rep2\FISH010.nd2,True,Cen7,rep2


In [52]:
f = px.box(df, color='Brightest', facet_col='Chromosome', x='rep', y='dR', points='all', width=900, facet_row='channel', height=500, title='Displacement from Nuclear Center by Chromosome and Replicate')
f.write_html('Distance_To_Center.html')
f

In [47]:
# Do Mann-Whitney U test on the dR values
from scipy.stats import mannwhitneyu
results = []
for chrom in df['Chromosome'].unique():
    for rep in df['rep'].unique():
        for channel in df['channel'].unique():
            sub = df[(df['Chromosome']==chrom) & (df['rep']==rep) & (df['channel']==channel)]
            group1 = sub[sub['Brightest']==True]['dR']
            group2 = sub[sub['Brightest']==False]['dR']
            if len(group1)>0 and len(group2)>0:
                stat, p = mannwhitneyu(group1, group2, alternative='two-sided')
                results.append({'Chromosome': chrom, 'rep': rep, 'channel': channel, 'group1_median': np.median(group1), 'group2_median': np.median(group2), 'U_statistic': stat, 'p_value': p, 'group1_n': len(group1), 'group2_n': len(group2)})

mann_df = pd.DataFrame(results)
mann_df.to_csv('Mann_Whitney_U_Results.csv')
mann_df

,Chromosome,rep,channel,group1_median,group2_median,U_statistic,p_value,group1_n,group2_n
0,Cen18,rep1,C0,4.356749,4.602379,1323.0,0.853006,52,52
1,Cen18,rep1,C1,3.691557,3.445239,1284.0,0.175870,47,47
2,Cen18,rep2,C0,5.367835,4.553161,1437.0,0.093576,49,49
3,Cen18,rep2,C1,4.317223,4.276042,1202.0,0.994331,49,49
4,Cen7,rep1,C0,4.451109,4.496540,2023.0,0.406814,61,61
5,Cen7,rep1,C1,4.187931,3.969229,1680.0,0.755272,57,57
6,Cen7,rep2,C0,5.217181,5.513335,525.0,0.519613,34,34
7,Cen7,rep2,C1,4.685803,4.384437,695.0,0.600490,36,36


In [53]:
f = px.box(df, color='Brightest', facet_col='Chromosome', x='rep', y='volume', points='all', width=900, range_y=[0,2000], height=500, facet_row='channel', title='Nuclear Volume by Chromosome and Replicate', hover_data=['fname', 'nucleus_label'])
f.write_html('Nuclear_Volume.html')
f

In [40]:
# Make interactive plot for examining kinteochore protein signal ratios
f=go.FigureWidget(
    px.box(df, x='rep', y='volume', color='Brightest', facet_col='Chromosome',  points='all', width=900, hover_data=['fname', 'nucleus_label'])
    )

def click_fn(trace, points, state):
    
    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        viewer.layers.clear()
        process_fname(idx, viewer=viewer, display=True)

for a in f.data:
    a.on_click(click_fn)
f

FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'boxpoints': 'all',
              'customdata': array([['Cen18-488_Cen6-561\\rep1\\FISH001.nd2', 1],
                                   ['Cen18-488_Cen6-561\\rep1\\FISH001.nd2', 2],
                                   ['Cen18-488_Cen6-561\\rep1\\FISH001.nd2', 3],
                                   ...,
                                   ['Cen18-488_Cen6-561\\rep2\\FISH010.nd2', 1],
                                   ['Cen18-488_Cen6-561\\rep2\\FISH010.nd2', 2],
                                   ['Cen18-488_Cen6-561\\rep2\\FISH010.nd2', 5]], dtype=object),
              'hovertemplate': ('Brightest=True<br>Chromosome=C' ... '{customdata[1]}<extra></extra>'),
              'legendgroup': 'True',
              'marker': {'color': '#636efa'},
              'name': 'True',
              'notched': False,
              'offsetgroup': 'True',
              'orientation': 'v',
              'showlegend': True,
              '

In [ ]:
f = px.box(df, color='Brightest', facet_col='Chromosome', x='rep', y='volume', points='all', width=900, range_y=[0,2000], height=500, facet_row='channel', title='Nuclear Volume by Chromosome and Replicate', hover_data=['fname', 'nucleus_label'])

In [58]:
df['DimBrightRatio'] = df.groupby(['fname', 'nucleus_label', 'channel'])['mean_intensity'].transform(lambda x: x.min()/x.max())

In [64]:
f = px.box(df, x='Chromosome', y='DimBrightRatio', color='rep', points='all', width=900, height=500, facet_row='channel', title='FISH Intensity Ratio', hover_data=['fname', 'nucleus_label'])
f.write_html('FISH_Dim_Bright_Intensity_Ratio.html')
f

In [65]:
fish_agged = df.groupby(['Chromosome', 'channel', 'rep']).agg({'DimBrightRatio': ['mean', 'std', 'count'],}).reset_index()
fish_agged.columns = ['Chromosome', 'channel', 'rep', 'mean_DimBrightRatio', 'std_DimBrightRatio', 'n']
fish_agged.to_csv('FISH_DimBrightRatio_Summary.csv')
fish_agged

,Chromosome,channel,rep,mean_DimBrightRatio,std_DimBrightRatio,n
0,Cen18,C0,rep1,0.866808,0.084014,104
1,Cen18,C0,rep2,0.867144,0.082041,98
2,Cen18,C1,rep1,0.781899,0.115059,94
3,Cen18,C1,rep2,0.787709,0.124479,98
4,Cen7,C0,rep1,0.849791,0.109218,122
5,Cen7,C0,rep2,0.866378,0.114862,68
6,Cen7,C1,rep1,0.676551,0.169778,114
7,Cen7,C1,rep2,0.683690,0.128734,72
